## Selecting baseline model

In [ ]:
import pandas as pd
import numpy as np
import warnings 
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [6]:
df = pd.read_csv('D:\ML_Projects\Loan_Approval\data\cleaned_loan_data.csv')
df

,person_age,person_gender,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status,person_education_Bachelor,person_education_Doctorate,person_education_High School,person_education_Master,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,22.0,0,71948.0,0,35000.0,16.02,0.49,3.0,561,0,1,0,0,0,1,0,0,1,0,0,0,1,0
1,21.0,0,12282.0,0,1000.0,11.14,0.08,2.0,504,1,0,0,0,1,0,0,1,0,1,0,0,0,0
2,25.0,0,12438.0,3,5500.0,12.87,0.44,3.0,635,0,1,0,0,1,0,0,0,0,0,0,1,0,0
3,23.0,0,79753.0,0,35000.0,15.23,0.44,2.0,675,0,1,1,0,0,0,0,0,1,0,0,1,0,0
4,24.0,1,66135.0,1,35000.0,14.27,0.53,4.0,586,0,1,0,0,0,1,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44988,27.0,1,47971.0,6,15000.0,15.66,0.31,3.0,645,0,1,0,0,0,0,0,0,1,0,0,1,0,0
44989,37.0,0,65800.0,17,9000.0,14.07,0.14,11.0,621,0,1,0,0,0,0,0,0,1,0,1,0,0,0
44990,33.0,1,56942.0,7,2771.0,10.02,0.05,10.0,668,0,1,0,0,0,0,0,0,1,0,0,0,0,0
44991,29.0,1,33164.0,4,12000.0,13.23,0.36,6.0,604,0,1,1,0,0,0,0,0,1,1,0,0,0,0


------>Splitting x and y<------

In [19]:
X = df.drop("loan_status",axis=1)
y = df['loan_status']
numerical_cols = X.select_dtypes(include='number').columns

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42,stratify=y)

scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])


------>Model selection<------

In [28]:
models = {
    "Logistic Regression" : LogisticRegression(random_state=42),
    "SVM" : SVC(random_state=42),
    "Random Forest" : RandomForestClassifier(random_state=42),
    "KNN" : KNeighborsClassifier(), 
    "Decision Tree" : DecisionTreeClassifier(random_state=42)
}

scoring = {
    "accuracy" : "accuracy",
    "precision" : "precision",
    "recall" : "recall",
    "f1" : "f1"
}

results = []

for name , model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=5,
        scoring=scoring
    )

    results.append({
        "Model" : name,
        "Accuracy" : scores['test_accuracy'].mean(),
        "precision" : scores['test_precision'].mean(),
        "recall" : scores['test_recall'].mean(),
        "F1_scores" : scores['test_f1'].mean()
    })

comparison = pd.DataFrame(results)
comparison =comparison.sort_values(
    by="F1_scores",
    ascending=False
)
comparison = comparison.round(3)
print(comparison)

                 Model  Accuracy  precision  recall  F1_scores
2        Random Forest     0.927      0.892   0.765      0.824
1                  SVM     0.914      0.844   0.750      0.794
4        Decision Tree     0.899      0.770   0.778      0.774
0  Logistic Regression     0.896      0.778   0.748      0.763
3                  KNN     0.891      0.794   0.688      0.737
